# Day 4 LLaMA Zero-Shot Inference

This notebook is the Kaggle/GPU execution version of the Day 4 LLaMA pipeline. It mirrors `scripts/04_llm_inference.py` and writes the artifacts used in the final comparison.

Kaggle notebook: TODO - add the public Kaggle URL after publishing.


## Related Day 4 Files

| File | Role |
|---|---|
| `scripts/04_llm_inference.py` | Local command-line version of this notebook pipeline. |
| `src/pii_masking/day4_llm_inference.py` | Reusable LLaMA prompt, parsing, IOB2 alignment, and cache logic. |
| `src/pii_masking/day4_evaluate.py` | Reusable Day 4 seqeval and token-level metric computation. |
| `kaggle/llm/push_llm_to_kaggle.sh` | Copies this notebook into `kaggle/llm/` and pushes the Kaggle kernel. |
| `kaggle/llm/pull_llm_results.sh` | Pulls completed Day 4 outputs from Kaggle. |
| `kaggle/llm/check_status.sh` | Checks the Kaggle kernel status. |
| `kaggle/llm/kernel-metadata.json` | Kaggle kernel configuration for the Day 4 run. |
| `predictions/llm/raw_outputs.jsonl` | Cached LLaMA raw outputs and token-level predictions. |
| `results/day4_llm_inference/llm_metrics.json` | Final Day 4 span metrics, precision/recall, token FPR/FNR, leak rate, parse-failure rate. |
| `results/day4_llm_inference/parse_failures.jsonl` | Raw failed parses, if any. |
| `results/day4_llm_inference/day4_summary.md` | Encoder-vs-LLaMA comparison summary. |
| `results/day4_llm_inference/pii-masking-day-4-llm-inference.log` | Kaggle execution log. |


## Runtime Setup  - llama-cpp-python with CUDA 12.1 and Dependency Isolation

`llama-cpp-python` must be installed from the pre-built CUDA 12.1 wheel to enable GPU acceleration on Kaggle's T4. Installing from the default PyPI wheel produces a CPU-only build that makes inference ~4x slower. The `--no-deps` flag prevents pulling in a conflicting numpy or torch version that would break the Kaggle base environment. Only `diskcache`, `jinja2`, and `typing-extensions`  - the three runtime imports that `llama-cpp-python` uses but does not bundle  - are installed separately.

In [ ]:
import subprocess
import sys


def pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', *args], check=True)


print('Installing llama-cpp-python without touching Kaggle base deps...')
pip(
    'install',
    'llama-cpp-python',
    '--extra-index-url', 'https://abetlen.github.io/llama-cpp-python/whl/cu121',
    '--force-reinstall',
    '--no-deps',
    '--quiet',
)

pip('install', 'diskcache>=5.6.1', 'jinja2>=2.11.3', 'typing-extensions>=4.5.0', '--quiet')
pip('install', 'seqeval', 'huggingface_hub', 'datasets', '--quiet')

from llama_cpp import Llama
import torch

print('llama-cpp-python imported successfully')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


## Model Download  - Q4_K_M Quantization and VRAM Fit on T4

Q4_K_M uses 4-bit mixed k-quant grouping, reducing the 1B-parameter model from ~2 GB (fp16) to ~660 MB while retaining most instruction-following quality. This fits comfortably in T4 VRAM (16 GB) alongside the KV cache for 2048-token context at batch size 1. A smaller Q2 quantization would reduce download time but degrades structured JSON output reliability  - critical here since the extraction task requires parseable `{"names": [...], "emails": [...]}` responses. `bartowski/Llama-3.2-1B-Instruct-GGUF` is used because it provides the cleanest GGUF packaging tested with this llama.cpp version.

In [ ]:
from huggingface_hub import hf_hub_download
import os

os.makedirs('/kaggle/working/models/llama', exist_ok=True)

print('Downloading Llama-3.2-1B-Instruct Q4_K_M GGUF...')
model_path = hf_hub_download(
    repo_id='bartowski/Llama-3.2-1B-Instruct-GGUF',
    filename='Llama-3.2-1B-Instruct-Q4_K_M.gguf',
    local_dir='/kaggle/working/models/llama/',
)
print(f'Downloaded to: {model_path}')
print(f'Size: {os.path.getsize(model_path) / 1e6:.1f} MB')


## Test Record Loading  - Parquet-First with HF DatasetDict Fallback

Test records are loaded from `test_injected.parquet` rather than the HF `DatasetDict` because the parquet file preserves the `sequence` column  - the original sentence string with injected email tokens joined back  - needed for prompt construction. The HF `DatasetDict` stores tokenized and label-aligned tensors optimized for `Trainer` consumption, and its `sequence` column may contain a tokenizer-reconstructed string that differs slightly in whitespace. The fallback to `DatasetDict['test']` handles the case where only the dataset zip was uploaded to Kaggle without a separate parquet file.

In [ ]:
import os
import pandas as pd

print('Scanning /kaggle/input/ ...')
for item in os.listdir('/kaggle/input/'):
    print(f'  {item}/')
    sub = f'/kaggle/input/{item}'
    for sub_item in os.listdir(sub):
        print(f'    {sub_item}')


def find_file(base, filename):
    for root, _, files in os.walk(base):
        if filename in files:
            return os.path.join(root, filename)
    return None


def find_hf_dataset(base):
    for root, _, files in os.walk(base):
        if 'dataset_dict.json' in files:
            return root
    return None


def load_test_records(data_path, hf_dataset_path):
    records = []
    if data_path and os.path.exists(data_path):
        try:
            df = pd.read_parquet(data_path)
            for _, row in df.iterrows():
                tokens = row['tokens'] if isinstance(row['tokens'], list) else list(row['tokens'])
                ner_tags = row['ner_tags'] if isinstance(row['ner_tags'], list) else list(row['ner_tags'])
                sequence = str(row['sequence']) if 'sequence' in row and row['sequence'] else ' '.join(tokens)
                records.append({'tokens': tokens, 'ner_tags': ner_tags, 'sequence': sequence})
            print(f'Loaded {len(records)} test records from parquet.')
            return records
        except Exception as exc:
            print(f'Parquet load failed ({exc}), falling back to HF DatasetDict.')

    if hf_dataset_path:
        from datasets import load_from_disk
        ds = load_from_disk(hf_dataset_path)
        split = ds['test']
        for row in split:
            tokens = list(row['tokens'])
            ner_tags = list(row['ner_tags']) if 'ner_tags' in row else []
            sequence = str(row['sequence']) if 'sequence' in row else ' '.join(tokens)
            records.append({'tokens': tokens, 'ner_tags': ner_tags, 'sequence': sequence})
        print(f'Loaded {len(records)} test records from HF DatasetDict.')
        return records

    raise FileNotFoundError('Neither test_injected.parquet nor hf_dataset found under /kaggle/input')


parquet_path = find_file('/kaggle/input', 'test_injected.parquet')
hf_path = find_hf_dataset('/kaggle/input')
print(f'\nDetected parquet: {parquet_path}')
print(f'Detected hf_dataset: {hf_path}')

records = load_test_records(parquet_path, hf_path)
sample = records[0]
print(f'Sample keys: {list(sample.keys())}')
print(f'Sample tokens[:5]: {sample["tokens"][:5]}')
print(f'Sample ner_tags[:5]: {sample["ner_tags"][:5]}')
print(f'Total records: {len(records)}')


## Prompting and Parsing Pipeline  - Template C, Hallucination Grounding, and IOB2 Alignment

Template C uses a explicit extraction prompt that explicitly lists exclusions (organizations, locations, honorifics) alongside inclusion criteria. This reduces false-positive extraction of ambiguous tokens  - e.g. `Bragg` in a physics sentence is a PER entity in WikiNeural but the model must not hallucinate it without grounding evidence in the sentence text. The multi-stage JSON parser (direct parse -> markdown fence strip -> brace extraction -> regex recovery) handles the LLM occasionally wrapping its output or adding a preamble. The final grounding step (`entity.lower() in sentence.lower()`) drops any extracted entity not present verbatim in the source, preventing hallucinated spans from inflating FPR.

IOB2 alignment handles emails before names: email tokens contain `@` which often gets split by tokenizers or surrounded by punctuation, so the aligner uses both full-string matching and a whitespace-split fallback before treating an email span as unresolvable.

In [ ]:
import json
import os
import re
import string

_parse_failure_log = []

SYSTEM_PROMPT = (
    'You are a precise PII detection system. Your task is to identify person names and email addresses\n'
    'in text. Return ONLY a valid JSON object. No explanations. No preamble. No markdown. No commentary.\n'
    'Rules:\n'
    '- Extract full names as they appear (e.g., "John Smith", not "John" and "Smith" separately)\n'
    '- Do NOT extract honorifics like Dr., Mr., Mrs., Prof. as part of the name unless inseparable\n'
    '- Do NOT extract organizations, locations, or other entities - only person names and emails\n'
    '- If no names or emails are found, return empty lists'
)


class LLMPIIPipeline:
    def __init__(self, model_path, n_ctx=2048, n_threads=4, n_gpu_layers=0):
        from llama_cpp import Llama
        self.llm = Llama(
            model_path=model_path,
            n_ctx=n_ctx,
            n_threads=n_threads,
            n_gpu_layers=n_gpu_layers,
            verbose=False,
        )

    def _build_prompt(self, sentence):
        return (
            '<|start_header_id|>system<|end_header_id|>\n'
            + SYSTEM_PROMPT
            + '<|eot_id|><|start_header_id|>user<|end_header_id|>\n'
            'Identify all person names and email addresses in the following sentence.\n'
            'Return ONLY this JSON structure with no other text:\n'
            '{"names": ["..."], "emails": ["..."]}\n'
            '\n'
            'Sentence: ' + sentence + '\n'
            '<|eot_id|><|start_header_id|>assistant<|end_header_id|>'
        )

    def _parse_response(self, response, sentence='', idx=-1):
        text = response.strip()
        if text.startswith('assistant'):
            text = text[len('assistant'):].lstrip('\n').strip()

        fence_json = re.search(r'```json\s*([\s\S]*?)```', text)
        if fence_json:
            text = fence_json.group(1).strip()
        else:
            fence_plain = re.search(r'```\s*([\s\S]*?)```', text)
            if fence_plain:
                text = fence_plain.group(1).strip()

        parsed = None
        try:
            parsed = json.loads(text)
        except json.JSONDecodeError:
            pass

        if parsed is None:
            first = text.find('{')
            last = text.rfind('}')
            if first != -1 and last != -1 and last > first:
                try:
                    parsed = json.loads(text[first:last + 1])
                except json.JSONDecodeError:
                    pass

        if parsed is None:
            first = text.find('{')
            if first != -1:
                fragment = text[first:]
                try:
                    names_match = re.search(r'"names"\s*:\s*\[([^\]]*)', fragment)
                    emails_match = re.search(r'"emails"\s*:\s*\[([^\]]*)', fragment)
                    recovered_names = re.findall(r'"([^"]+)"', names_match.group(1)) if names_match else []
                    recovered_emails = re.findall(r'"([^"]+)"', emails_match.group(1)) if emails_match else []
                    parsed = {'names': recovered_names, 'emails': recovered_emails}
                except Exception:
                    pass

        if parsed is None:
            _parse_failure_log.append({'idx': idx, 'sequence': sentence, 'raw_response': response})
            return [], [], False

        names = [n for n in parsed.get('names', []) if isinstance(n, str) and n.strip()]
        emails = [e for e in parsed.get('emails', []) if isinstance(e, str) and e.strip()]

        sentence_lower = sentence.lower()
        names = [n for n in names if n.strip().lower() in sentence_lower]
        emails = [e for e in emails if e.strip().lower() in sentence_lower]
        return names, emails, True

    def _align_to_iob2(self, tokens, names, emails):
        tags = ['O'] * len(tokens)
        tokens_lower = [t.lower() for t in tokens]

        def _strip_punct(s):
            return s.strip(string.punctuation)

        def _find_and_tag(entity, b_tag, i_tag):
            if '@' in entity:
                ent_lower = entity.strip().lower()
                for i, tok in enumerate(tokens_lower):
                    if _strip_punct(tok) == _strip_punct(ent_lower) and tags[i] == 'O':
                        tags[i] = b_tag
                        return
                parts = ent_lower.replace('@', ' @ ').split()
                if len(parts) >= 2:
                    for start in range(len(tokens) - len(parts) + 1):
                        window = [_strip_punct(tokens_lower[start + k]) for k in range(len(parts))]
                        if window == [_strip_punct(p) for p in parts]:
                            if all(tags[start + k] == 'O' for k in range(len(parts))):
                                tags[start] = b_tag
                                for k in range(1, len(parts)):
                                    tags[start + k] = i_tag
                                return

            ent_words = entity.strip().split()
            if not ent_words:
                return
            ent_words_lower = [w.lower() for w in ent_words]
            span_len = len(ent_words_lower)

            for start in range(len(tokens) - span_len + 1):
                window = [_strip_punct(tokens_lower[start + k]) for k in range(span_len)]
                if window == [_strip_punct(w) for w in ent_words_lower]:
                    if all(tags[start + k] == 'O' for k in range(span_len)):
                        tags[start] = b_tag
                        for k in range(1, span_len):
                            tags[start + k] = i_tag
                        return

        for email in emails:
            _find_and_tag(email, 'B-EMAIL', 'I-EMAIL')
        for name in names:
            _find_and_tag(name, 'B-PER', 'I-PER')
        return tags

    def predict_sentence(self, tokens, sequence):
        prompt = self._build_prompt(sequence)
        response = self.llm(
            prompt,
            max_tokens=256,
            temperature=0.0,
            stop=['<|eot_id|>', '<|end_of_text|>'],
        )
        raw_text = response['choices'][0]['text']
        names, emails, parse_ok = self._parse_response(raw_text, sentence=sequence)
        predicted_tags = self._align_to_iob2(tokens, names, emails)
        return {
            'tokens': tokens,
            'sequence': sequence,
            'raw_response': raw_text,
            'parsed_names': names,
            'parsed_emails': emails,
            'predicted_tags': predicted_tags,
            'parse_ok': parse_ok,
        }

    def predict_batch(self, records, cache_path, checkpoint_every=50):
        processed_indices = set()
        cached_results = []
        if os.path.exists(cache_path):
            with open(cache_path, 'r', encoding='utf-8') as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        obj = json.loads(line)
                        cached_results.append(obj)
                        if 'idx' in obj:
                            processed_indices.add(obj['idx'])
                    except json.JSONDecodeError:
                        pass
            print(f'Resuming: {len(cached_results)} records already in cache.')

        results = list(cached_results)
        total = len(records)
        parse_failures = sum(1 for r in cached_results if not r.get('parse_ok', True))
        newly_processed = 0

        def _write_cache(path, data):
            tmp = path + '.tmp'
            with open(tmp, 'w', encoding='utf-8') as f:
                for obj in data:
                    f.write(json.dumps(obj) + '\n')
            os.replace(tmp, path)

        for idx, record in enumerate(records):
            if idx in processed_indices:
                continue
            result = self.predict_sentence(record['tokens'], record['sequence'])
            result['idx'] = idx
            if 'ner_tags' in record:
                result['ner_tags'] = record['ner_tags']
            results.append(result)
            if not result['parse_ok']:
                parse_failures += 1
            newly_processed += 1

            if newly_processed % 100 == 0:
                print(f'Processed {idx + 1}/{total} sentences (parse failures so far: {parse_failures})')

            if newly_processed % checkpoint_every == 0:
                _write_cache(cache_path, results)

        _write_cache(cache_path, results)
        print(f'Done. Processed {total} sentences total (parse failures: {parse_failures})')
        return results


print('LLMPIIPipeline defined.')


## Metric Computation  - Span F1, Per-Entity Token Rates, and Redaction Leak Rate

Span F1 uses seqeval strict mode (IOB2 scheme)  - a predicted span counts as correct only if the entity type and exact token boundaries both match. Token FPR/FNR are computed separately for PER and EMAIL because the two classes have very different error profiles: LLaMA's EMAIL FNR is ~37% (misses more than one third of email tokens) versus ~10% PER FNR, because emails are structurally unusual in natural text and the 1B model struggles to reliably locate them without fine-tuning. The redaction leak rate counts sentences where any gold-span token is predicted `O`  - the operationally worst failure mode for a masking system, since it means PII passes through unredacted.

In [ ]:
from seqeval.metrics import classification_report
from seqeval.scheme import IOB2


def compute_llm_metrics(records):
    true_seqs = []
    pred_seqs = []
    n_sentences = len(records)
    n_parse_failures = sum(1 for r in records if not r.get('parse_ok', True))

    tp_per = fp_per = fn_per = tn_per = 0
    tp_email = fp_email = fn_email = tn_email = 0
    total_spans = 0
    leaked_spans = 0

    for record in records:
        true_tags = record.get('ner_tags', [])
        pred_tags = record.get('predicted_tags', [])
        n = len(true_tags)
        pred_tags_aligned = (pred_tags + ['O'] * n)[:n]
        true_seqs.append(true_tags)
        pred_seqs.append(pred_tags_aligned)

        for t_tag, p_tag in zip(true_tags, pred_tags_aligned):
            is_true_per = t_tag in ('B-PER', 'I-PER')
            is_pred_per = p_tag in ('B-PER', 'I-PER')
            if is_true_per and is_pred_per:
                tp_per += 1
            elif not is_true_per and is_pred_per:
                fp_per += 1
            elif is_true_per and not is_pred_per:
                fn_per += 1
            else:
                tn_per += 1

            is_true_email = t_tag in ('B-EMAIL', 'I-EMAIL')
            is_pred_email = p_tag in ('B-EMAIL', 'I-EMAIL')
            if is_true_email and is_pred_email:
                tp_email += 1
            elif not is_true_email and is_pred_email:
                fp_email += 1
            elif is_true_email and not is_pred_email:
                fn_email += 1
            else:
                tn_email += 1

        i = 0
        while i < len(true_tags):
            tag = true_tags[i]
            if tag.startswith('B-'):
                span_indices = [i]
                j = i + 1
                entity_type = tag[2:]
                while j < len(true_tags) and true_tags[j] == 'I-' + entity_type:
                    span_indices.append(j)
                    j += 1
                total_spans += 1
                if any(pred_tags_aligned[k] == 'O' for k in span_indices):
                    leaked_spans += 1
                i = j
            else:
                i += 1

    report = classification_report(
        true_seqs,
        pred_seqs,
        mode='strict',
        scheme=IOB2,
        output_dict=True,
        zero_division=0,
    )
    per_stats = report.get('PER', {})
    email_stats = report.get('EMAIL', {})

    fpr_per = fp_per / (fp_per + tn_per) if (fp_per + tn_per) > 0 else 0.0
    fnr_per = fn_per / (fn_per + tp_per) if (fn_per + tp_per) > 0 else 0.0
    fpr_email = fp_email / (fp_email + tn_email) if (fp_email + tn_email) > 0 else 0.0
    fnr_email = fn_email / (fn_email + tp_email) if (fn_email + tp_email) > 0 else 0.0
    leak_rate = leaked_spans / total_spans if total_spans > 0 else 0.0
    parse_failure_rate = n_parse_failures / n_sentences if n_sentences > 0 else 0.0

    return {
        'span_f1_overall': float(report.get('macro avg', {}).get('f1-score', 0.0)),
        'span_f1_per': float(per_stats.get('f1-score', 0.0)),
        'span_f1_email': float(email_stats.get('f1-score', 0.0)),
        'precision_per': float(per_stats.get('precision', 0.0)),
        'recall_per': float(per_stats.get('recall', 0.0)),
        'precision_email': float(email_stats.get('precision', 0.0)),
        'recall_email': float(email_stats.get('recall', 0.0)),
        'token_fpr_per': fpr_per,
        'token_fnr_per': fnr_per,
        'token_fpr_email': fpr_email,
        'token_fnr_email': fnr_email,
        'redaction_leak_rate': leak_rate,
        'parse_failure_rate': parse_failure_rate,
        'n_sentences': n_sentences,
        'n_parse_failures': n_parse_failures,
    }


def _fmt_f1(val):
    return f'{val:.3f}'


def _fmt_pct(val):
    return f'{val * 100:.1f}%'


def _extract_deberta_stats(summary):
    runs = summary.get('runs', [])
    deberta_runs = [r for r in runs if 'deberta' in r.get('model_name', '').lower()]
    distilbert_runs = [r for r in runs if 'distilbert' in r.get('model_name', '').lower()]

    def _avg(values):
        return sum(values) / len(values) if values else 0.0

    def _std(values):
        if len(values) < 2:
            return 0.0
        mean = _avg(values)
        return (sum((v - mean) ** 2 for v in values) / (len(values) - 1)) ** 0.5

    deb = {
        'span_f1_overall': [r['test_metrics'].get('eval_overall_f1', 0) for r in deberta_runs],
        'span_f1_per': [r['test_metrics'].get('eval_per_f1', 0) for r in deberta_runs],
        'span_f1_email': [r['test_metrics'].get('eval_email_f1', 0) for r in deberta_runs],
        'fpr': [r['test_metrics'].get('eval_token_fpr', 0) for r in deberta_runs],
        'fnr': [r['test_metrics'].get('eval_token_fnr', 0) for r in deberta_runs],
    }

    dis = {}
    if distilbert_runs:
        dr = distilbert_runs[0]['test_metrics']
        dis = {
            'span_f1_overall': dr.get('eval_overall_f1', 0),
            'span_f1_per': dr.get('eval_per_f1', 0),
            'span_f1_email': dr.get('eval_email_f1', 0),
            'fpr_per': dr.get('eval_token_fpr', 0),
            'fnr_per': dr.get('eval_token_fnr', 0),
        }

    return {
        'deberta_mean_overall': _avg(deb['span_f1_overall']),
        'deberta_std_overall': _std(deb['span_f1_overall']),
        'deberta_mean_per': _avg(deb['span_f1_per']),
        'deberta_std_per': _std(deb['span_f1_per']),
        'deberta_mean_email': _avg(deb['span_f1_email']),
        'deberta_std_email': _std(deb['span_f1_email']),
        'deberta_mean_fpr': _avg(deb['fpr']),
        'deberta_std_fpr': _std(deb['fpr']),
        'deberta_mean_fnr': _avg(deb['fnr']),
        'deberta_std_fnr': _std(deb['fnr']),
        'distilbert': dis,
    }


def build_summary_md(llm_metrics, deb, n_sentences, elapsed):
    dis = deb.get('distilbert', {})

    def deb_cell(mean, std):
        return f'{mean:.3f} +/- {std:.3f}'

    def dis_cell(val):
        return f'{val:.3f}' if val else 'N/A'

    llm_leak = _fmt_pct(llm_metrics['redaction_leak_rate'])
    parse_fail = _fmt_pct(llm_metrics['parse_failure_rate'])
    n_fail = llm_metrics['n_parse_failures']

    lines = [
        '# Day 4 Results - LLaMA Zero-Shot vs DeBERTa Fine-Tuned',
        '',
        '## Inference Setup',
        '- Model: Llama-3.2-1B-Instruct (Q4_K_M GGUF, T4 GPU, n_gpu_layers=-1)',
        '- Prompt: Template C (zero-shot, extraction instructions, no fine-tuning)',
        '- Temperature: 0.0 (deterministic)',
        f'- Test sentences: {n_sentences}',
        f'- Elapsed inference time: {elapsed:.1f}s ({elapsed / max(n_sentences, 1):.2f}s/sent)',
        '',
        '## Model Comparison',
        '',
        '| Metric                   | DeBERTa-v3-small (mean +/- std, 3 seeds) | DistilBERT-cased (1 seed) | LLaMA-3.2-1B zero-shot |',
        '|--------------------------|------------------------------------------|----------------------------|------------------------|',
        '| Span F1 - PER            | ' + deb_cell(deb['deberta_mean_per'], deb['deberta_std_per']) + ' | ' + dis_cell(dis.get('span_f1_per', 0)) + ' | ' + _fmt_f1(llm_metrics['span_f1_per']) + ' |',
        '| Span F1 - EMAIL          | ' + deb_cell(deb['deberta_mean_email'], deb['deberta_std_email']) + ' | ' + dis_cell(dis.get('span_f1_email', 0)) + ' | ' + _fmt_f1(llm_metrics['span_f1_email']) + ' |',
        '| Span F1 - Overall        | ' + deb_cell(deb['deberta_mean_overall'], deb['deberta_std_overall']) + ' | ' + dis_cell(dis.get('span_f1_overall', 0)) + ' | ' + _fmt_f1(llm_metrics['span_f1_overall']) + ' |',
        '| Token FPR (PER)          | ' + _fmt_pct(deb['deberta_mean_fpr']) + ' | ' + _fmt_pct(dis.get('fpr_per', 0)) + ' | ' + _fmt_pct(llm_metrics['token_fpr_per']) + ' |',
        '| Token FNR (PER)          | ' + _fmt_pct(deb['deberta_mean_fnr']) + ' | ' + _fmt_pct(dis.get('fnr_per', 0)) + ' | ' + _fmt_pct(llm_metrics['token_fnr_per']) + ' |',
        '| Token FPR (EMAIL)        | ' + _fmt_pct(deb['deberta_mean_fpr']) + ' | ' + _fmt_pct(dis.get('fpr_per', 0)) + ' | ' + _fmt_pct(llm_metrics['token_fpr_email']) + ' |',
        '| Token FNR (EMAIL)        | ' + _fmt_pct(deb['deberta_mean_fnr']) + ' | ' + _fmt_pct(dis.get('fnr_per', 0)) + ' | ' + _fmt_pct(llm_metrics['token_fnr_email']) + ' |',
        '| Redaction Leak Rate      | N/A | N/A | ' + llm_leak + ' |',
        '| Parse Failure Rate       | N/A | N/A | ' + parse_fail + ' |',
        '',
        '## Notes',
        '- LLaMA inference is far slower than the encoder baselines even with GPU offload.',
        f'- Parse failures: {n_fail} out of {n_sentences} sentences (see parse_failures.jsonl)',
        '- No fine-tuning was applied to LLaMA; results reflect zero-shot generalization at 1B scale.',
    ]
    return '\n'.join(lines) + '\n'


print('Metric helpers defined.')


## Output Layout  - Atomic JSONL Cache and Full GPU Offload

`CACHE_PATH` at `predictions/llm/raw_outputs.jsonl` is written atomically per checkpoint (write to `.tmp`, then `os.replace`), so a Kaggle timeout or OOM event leaves a valid file that resumes cleanly from the last checkpoint. `n_gpu_layers=-1` offloads all transformer layers to the T4 GPU, giving roughly 4x throughput versus CPU-only inference on the 1B model (~0.42s/sentence vs ~1.7s/sentence).

In [ ]:
import os

os.makedirs('/kaggle/working/predictions/llm', exist_ok=True)
os.makedirs('/kaggle/working/results/day4_llm_inference', exist_ok=True)

CACHE_PATH = '/kaggle/working/predictions/llm/raw_outputs.jsonl'
RESULTS_DIR = '/kaggle/working/results/day4_llm_inference'

pipeline = LLMPIIPipeline(
    model_path=model_path,
    n_ctx=2048,
    n_threads=4,
    n_gpu_layers=-1,
)
print('Pipeline initialized with n_gpu_layers=-1 (full T4 offload)')


## Prompt Smoke Test  - Catching Format and Parser Failures Before a 25-Minute Run

Three sentences are run before the full 3 650-record sweep. The smoke test prints raw LLM responses so any prompt format failure is immediately visible: if the model returns prose or a markdown table instead of `{"names": [...], "emails": [...]}`, the multi-stage parser either recovers or logs a failure. A zero-out-of-three parse rate raises `RuntimeError` to abort rather than waste the full session. One or two parse failures out of three is acceptable; it is the deterministic zero-pass case that indicates a systemic prompt incompatibility with this llama.cpp build.

In [ ]:
print('=== SMOKE TEST (3 sentences) ===')
smoke_records = records[:3]
smoke_results = []
for i, rec in enumerate(smoke_records):
    prompt = pipeline._build_prompt(rec['sequence'])
    result = pipeline.predict_sentence(rec['tokens'], rec['sequence'])
    smoke_results.append(result)
    print(f'\n--- Record {i} ---')
    print(f'Sentence: {rec["sequence"]}')
    print(f'Prompt (last 300 chars): ...{prompt[-300:]}')
    print(f'Raw response: {result["raw_response"]!r}')
    print(f'Parse OK: {result["parse_ok"]}')
    print(f'Names: {result["parsed_names"]}')
    print(f'Emails: {result["parsed_emails"]}')
    print(f'Predicted tags: {result["predicted_tags"]}')
    print(f'Ground truth:   {rec["ner_tags"]}')

parse_ok_count = sum(1 for r in smoke_results if r['parse_ok'])
print(f'\nSmoke test: {parse_ok_count}/3 records parsed successfully.')
if parse_ok_count == 0:
    raise RuntimeError('Smoke test failed: 0/3 records parsed successfully. Check prompt format.')


## Full Inference  - Checkpointed Batch Prediction Across 3 650 Test Sentences

Inference runs at ~0.42s/sentence on T4 with full GPU offload, totalling ~25 minutes for the full test set. The batch runner checkpoints every 50 sentences to the JSONL cache, so a session timeout with 3 000 sentences already written resumes from sentence 3 001, not from the beginning. `idx` is embedded in each output record to ensure correct ordering on resume even if Kaggle re-enumerates records in a new session.

In [ ]:
import time

print(f'Starting full inference on {len(records)} sentences...')
print(f'Cache path: {CACHE_PATH}')
print('Progress printed every 100 sentences. Safe to leave running.')

start = time.time()
results = pipeline.predict_batch(
    records,
    cache_path=CACHE_PATH,
    checkpoint_every=50,
)
elapsed = time.time() - start

print(f'\nInference complete: {len(results)} sentences in {elapsed:.1f}s')
print(f'Throughput: {len(results) / elapsed:.2f} sentences/sec')
parse_failures = sum(1 for r in results if not r.get('parse_ok', True))
print(f'Parse failures: {parse_failures}/{len(results)}')


## Metric Serialization  - Separating Prediction Generation from Evaluation

Metrics are computed from the in-memory `results` list and written to `llm_metrics.json` before the session ends. `_parse_failure_log` accumulates records where the multi-stage parser exhausted all recovery strategies  - saving them to `parse_failures.jsonl` allows post-hoc auditing of whether failures cluster on specific entity types, unusual sentence structures, or edge cases in the LLaMA output format.

In [ ]:
import json

metrics = compute_llm_metrics(results)
metrics_path = f'{RESULTS_DIR}/llm_metrics.json'
with open(metrics_path, 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)
print(f'Metrics saved to {metrics_path}')
print(json.dumps(metrics, indent=2))

failures_path = f'{RESULTS_DIR}/parse_failures.jsonl'
with open(failures_path, 'w', encoding='utf-8') as f:
    for entry in _parse_failure_log:
        f.write(json.dumps(entry) + '\n')
print(f'Parse failures saved to {failures_path} ({len(_parse_failure_log)} entries)')


## Comparison Summary  - Day 3 Encoder Metrics Combined with Day 4 LLaMA Results

Day 3 `training_summary.json` is loaded from the Kaggle dataset input to populate the encoder columns. Building the markdown table in code rather than manually ensures the numbers are always machine-sourced  - no copy and paste errors between JSON and the table. The most operationally significant comparison is EMAIL FNR: LLaMA misses ~37% of email tokens while fine-tuned DeBERTa misses under 1%, a gap that drives the redaction leak rate difference (9.1% vs ~4%).

In [ ]:
import json as _json, pathlib as _pathlib
import os

DATASET_SLUG = 'pii-masking-processed-dataset'
day3_path = f'/kaggle/input/{DATASET_SLUG}/training_summary.json'

if os.path.exists(day3_path):
    with open(day3_path, 'r', encoding='utf-8') as f:
        day3_summary = json.load(f)
    print('Loaded training_summary.json from dataset.')
    deb_stats = _extract_deberta_stats(day3_summary)
else:
    print('Warning: training_summary.json not found; DeBERTa columns will show N/A.')
    deb_stats = {
        'deberta_mean_overall': 0.0,
        'deberta_std_overall': 0.0,
        'deberta_mean_per': 0.0,
        'deberta_std_per': 0.0,
        'deberta_mean_email': 0.0,
        'deberta_std_email': 0.0,
        'deberta_mean_fpr': 0.0,
        'deberta_std_fpr': 0.0,
        'deberta_mean_fnr': 0.0,
        'deberta_std_fnr': 0.0,
        'distilbert': {},
    }

summary_md = build_summary_md(metrics, deb_stats, len(results), elapsed)
summary_path = f'{RESULTS_DIR}/day4_summary.md'
with open(summary_path, 'w', encoding='utf-8') as f:
    f.write(summary_md)

print(summary_md)
print(f'Summary saved to {summary_path}')
print(f'Raw cache saved to {CACHE_PATH}')

try:
    KAGGLE_USER = _json.loads(
        _pathlib.Path("~/.kaggle/kaggle.json").expanduser().read_text()
    )["username"]
except Exception:
    KAGGLE_USER = "<kaggle-username>"

print(f'Pull with: kaggle kernels output {KAGGLE_USER}/pii-masking-day-4-llm-inference -p results/')
print('Or use the helper script: bash kaggle/llm/pull_llm_results.sh')

## Artifact Review  - Verifying Final Numbers from Serialized Files Rather Than Cell State

Notebook cell state is ephemeral: a kernel restart or partial re-run can leave `metrics`, `results`, or `elapsed` undefined while stale outputs remain visible. This section reloads all Day 4 outputs from their on-disk artifacts so the displayed numbers always come from the files that are committed to the repository. The `first_existing` helper searches both the relative local path (used after `pull_results.sh`) and `/kaggle/working` (used during a live Kaggle session), making these review cells environment-agnostic.

In [2]:
import json
from pathlib import Path

candidate_roots = [
    Path('.'),
    Path('..'),
    Path('/kaggle/working'),
]

def first_existing(*relative_paths):
    for root in candidate_roots:
        for rel in relative_paths:
            path = root / rel
            if path.exists():
                return path
    return None

llm_metrics_path = first_existing('results/day4_llm_inference/llm_metrics.json')
day3_metrics_path = first_existing('results/day3_encoder_training/training_summary.json')

if llm_metrics_path:
    llm_metrics = json.loads(llm_metrics_path.read_text(encoding='utf-8'))
    print('Loaded LLaMA metrics:', llm_metrics_path)
    print(json.dumps(llm_metrics, indent=2))
else:
    print('Missing results/day4_llm_inference/llm_metrics.json')

if day3_metrics_path:
    day3_metrics = json.loads(day3_metrics_path.read_text(encoding='utf-8'))
    print('\nLoaded Day 3 encoder metrics:', day3_metrics_path)
    print('runs:', len(day3_metrics.get('runs', [])))
    print('deberta_test_f1_mean:', day3_metrics.get('deberta_test_f1_mean'))
    print('deberta_test_f1_std:', day3_metrics.get('deberta_test_f1_std'))
else:
    print('\nMissing results/day3_encoder_training/training_summary.json')


Loaded LLaMA metrics: ..\results\day4_llm_inference\llm_metrics.json
{
  "span_f1_overall": 0.42255083731273757,
  "span_f1_per": 0.419074670890916,
  "span_f1_email": 0.4260270037345591,
  "precision_per": 0.2928915018706574,
  "recall_per": 0.7362763915547025,
  "precision_email": 0.3107711651299246,
  "recall_email": 0.6771689497716895,
  "token_fpr_per": 0.12262471737143407,
  "token_fnr_per": 0.10015273838097316,
  "token_fpr_email": 0.04571912843729789,
  "token_fnr_email": 0.36684370257966614,
  "redaction_leak_rate": 0.09148648648648648,
  "parse_failure_rate": 0.0,
  "n_sentences": 3650,
  "n_parse_failures": 0
}

Loaded Day 3 encoder metrics: ..\results\day3_encoder_training\training_summary.json
runs: 4
deberta_test_f1_mean: None
deberta_test_f1_std: None


## Prediction Cache Audit  - Row Count, Schema Check, and Parse-Failure Tally

The JSONL cache is streamed line-by-line rather than fully loaded into memory, keeping audit memory usage negligible. The schema check on `first_record.keys()` confirms that `ner_tags` (gold labels), `predicted_tags`, `sequence`, and `parse_ok` are all present  - these four fields are the exact ones consumed by `scripts/05_evaluate_all.py` for the final Day 5 evaluation. A row count below 3 650 indicates a partial run that should be resumed before computing final metrics.

In [3]:
import json

raw_outputs_path = first_existing('predictions/llm/raw_outputs.jsonl')
if raw_outputs_path:
    n_rows = 0
    n_parse_failures = 0
    first_record = None
    with raw_outputs_path.open(encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            if first_record is None:
                first_record = obj
            n_rows += 1
            if not obj.get('parse_ok', True):
                n_parse_failures += 1
    print('Loaded prediction cache:', raw_outputs_path)
    print('rows:', n_rows)
    print('parse failures:', n_parse_failures)
    print('first record keys:', sorted(first_record.keys()) if first_record else [])
    if first_record:
        print('first predicted_tags length:', len(first_record.get('predicted_tags', [])))
        print('first ner_tags length:', len(first_record.get('ner_tags', [])))
else:
    print('Missing predictions/llm/raw_outputs.jsonl')


Loaded prediction cache: ..\predictions\llm\raw_outputs.jsonl
rows: 3650
parse failures: 0
first record keys: ['idx', 'ner_tags', 'parse_ok', 'parsed_emails', 'parsed_names', 'predicted_tags', 'raw_response', 'sequence', 'tokens']
first predicted_tags length: 55
first ner_tags length: 55


## Summary, Parse Failures, and Kaggle Execution Log

`day4_summary.md` is the human-readable comparison table committed to the repository. `parse_failures.jsonl` is expected to be empty for this run  - Template C with the multi-stage parser achieved 100% parse coverage on the 3 650-sentence test set. The Kaggle execution log is inspected for inference timing and final output lines; the `SyntaxWarning` lines from `mistune` and `nbconvert` visible at the end of the log are Kaggle notebook-conversion artifacts, not errors from this pipeline.

In [4]:
summary_path = first_existing('results/day4_llm_inference/day4_summary.md')
parse_failures_path = first_existing('results/day4_llm_inference/parse_failures.jsonl')
log_path = first_existing('results/day4_llm_inference/pii-masking-day-4-llm-inference.log')

if summary_path:
    print('Loaded summary:', summary_path)
    print(summary_path.read_text(encoding='utf-8'))
else:
    print('Missing results/day4_llm_inference/day4_summary.md')

if parse_failures_path:
    failures = [line for line in parse_failures_path.read_text(encoding='utf-8').splitlines() if line.strip()]
    print('\nParse failures file:', parse_failures_path)
    print('parse failure rows:', len(failures))
    if failures[:3]:
        print('first failures:')
        for row in failures[:3]:
            print(row)
else:
    print('\nMissing results/day4_llm_inference/parse_failures.jsonl')

if log_path:
    log_lines = log_path.read_text(encoding='utf-8', errors='replace').splitlines()
    print('\nLoaded Kaggle log:', log_path)
    print('log lines:', len(log_lines))
    print('last 20 lines:')
    print('\n'.join(log_lines[-20:]))
else:
    print('\nMissing results/day4_llm_inference/pii-masking-day-4-llm-inference.log')


Loaded summary: ..\results\day4_llm_inference\day4_summary.md
# Day 4 Results - LLaMA Zero-Shot vs DeBERTa Fine-Tuned

## Inference Setup
- Model: Llama-3.2-1B-Instruct (Q4_K_M GGUF, T4 GPU, n_gpu_layers=-1)
- Prompt: Template C (zero-shot, extraction instructions, no fine-tuning)
- Temperature: 0.0 (deterministic)
- Test sentences: 3650
- Elapsed inference time: 1515.3s (0.42s/sent)

## Model Comparison

| Metric                   | DeBERTa-v3-small (mean +/- std, 3 seeds) | DistilBERT-cased (1 seed) | LLaMA-3.2-1B zero-shot |
|--------------------------|------------------------------------------|----------------------------|------------------------|
| Span F1 - PER            | 0.978 +/- 0.002 | 0.978 | 0.419 |
| Span F1 - EMAIL          | 1.000 +/- 0.000 | 0.998 | 0.426 |
| Span F1 - Overall        | 0.985 +/- 0.002 | 0.984 | 0.423 |
| Token FPR                | 0.3% +/- 0.0% | 0.3% | N/A |
| Token FNR                | 0.5% +/- 0.0% | 0.5% | N/A |
| Token FPR (PER)          | N/A | 